# CSEE 4121 — In-class workshop on security & ML systems

We have 5 hands-on exercises here in two sections:

- Exercises 1–3: SQL injection, password storage, and data anonymization
- Exercises 4–5: GPU memory math and profiling training loops

**Instructions:**
- Work through the exercises in order. Each one builds intuition for the next.
- Look for cells marked with `# TODO` or `YOUR CODE HERE`. Fill in those parts.
- Cells that say "Your answer:" are short-answer questions. Type your response in the cell.
- If you get stuck, ask for help.

Let's get started!


In [ ]:
# =============================================
# Setup — make sure you have all needed modules
# =============================================
!pip install bcrypt -q

import sqlite3
import hashlib
import time
import os
import pandas as pd
import numpy as np

print("All imports successful. Ready to go!")


---
## Security & Privacy

---

### Exercise 1: SQL Injection

SQL injection is one of the oldest and most common web vulnerabilities. It happens when a backend application builds SQL queries by directly concatenating user input into the query string.
An attacker can craft special input that changes the meaning of the query — potentially bypassing login, stealing data, or even destroying the database.

In this exercise, you'll be the attacker first to understand the threat, and then be the defender to learn how to prevent it.


In [ ]:
# =================================================
# Exercise 1 Setup — create a database
# =================================================
#
# Simulates a database for a small web application.
# DO NOT MODIFY

import sqlite3
import hashlib

def setup_database():
    """Create an in-memory SQLite database with a users table."""
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            username TEXT UNIQUE,
            password TEXT,
            email TEXT,
            role TEXT,
            credit_card TEXT
        )
    """)

    # Insert some users (passwords stored in plaintext — this is bad! but realistic for this exercise)
    users = [
        ("alice",   "password123",   "alice@example.com",   "admin",   "4111-1111-1111-1111"),
        ("bob",     "letmein",       "bob@example.com",     "user",    "4222-2222-2222-2222"),
        ("charlie", "qwerty2024",    "charlie@example.com", "user",    "4333-3333-3333-3333"),
        ("diana",   "securePass!1",  "diana@example.com",   "analyst", "4444-4444-4444-4444"),
        ("eve",     "trustno1",      "eve@example.com",     "user",    "4555-5555-5555-5555"),
        ("frank",   "hunter2",       "frank@example.com",   "user",    "4666-6666-6666-6666"),
        ("grace",   "ilovecats",     "grace@example.com",   "analyst", "4777-7777-7777-7777"),
        ("heidi",   "P@ssw0rd",      "heidi@example.com",   "admin",   "4888-8888-8888-8888"),
        ("ivan",    "dragon99",      "ivan@example.com",    "user",    "4999-9999-9999-9999"),
        ("judy",    "welcome1",      "judy@example.com",    "user",    "4000-0000-0000-0000"),
    ]

    for u in users:
        cursor.execute("INSERT INTO users (username, password, email, role, credit_card) VALUES (?,?,?,?,?)", u)

    conn.commit()
    return conn

# Create the database
db = setup_database()

def run_query(conn, query):
    try:
        cursor = conn.cursor()
        cursor.execute(query)
        rows = cursor.fetchall()
        if rows:
            # Get column names
            col_names = [desc[0] for desc in cursor.description] if cursor.description else []
            print(f"Columns: {col_names}")
            for row in rows:
                print(row)
            print(f"\n{len(rows)} row(s) returned")
        else:
            print("No rows returned")
        return rows
    except Exception as e:
        print(f"Error: {e}")
        return []

print("Database created with 10 users.")
print("Table: users (id, username, password, email, role, credit_card)")


#### Task 1a: Exploiting a Vulnerable Login Function

Below is a function that a (careless) developer might write. It takes a username and password from a web form, builds a SQL query using string concatenation, and checks if any matching row exists.

Read the function carefully. Notice how `username` and `password` are dropped directly into the SQL string using f-string formatting. This is a vulnerability.


In [ ]:
# The vulnerable login function
# DO NOT MODIFY

def login_vulnerable(conn, username, password):
    """
    Returns True if login succeeds, plus any rows returned.
    """
    query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"
    print(f"Generated query:\n  {query}\n")

    cursor = conn.cursor()
    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        if rows:
            print(f"Login succeeded! Returned {len(rows)} row(s):")
            for row in rows:
                print(f"  {row}")
            return True, rows
        else:
            print("Login failed. No matching user.")
            return False, []
    except Exception as e:
        print(f"Error: {e}")
        return False, []

print("=== Login attempt ===")
login_vulnerable(db, "alice", "password123")


#### Let's attack this login

For each attack below, you need to fill in the `username` and/or `password` strings with specially crafted inputs that exploit the string concatenation vulnerability.

**Hints:**
- Think about what the final SQL query looks like after your input is substituted in.
- `'` (single quote), `OR` (disjunction), `--` (SQL comment), `;` (statement separator) might be helpful.
- Disjunction with true is a classic pattern. Think about why it works.
- `UNION SELECT` lets you combine results from different queries.


In [ ]:
# ========================================
# TODO: Attack 1 — Bypass the login
# ========================================
# Goal: Log in without knowing any password.
# Craft a username that makes the WHERE clause always true.

attack1_username = ""  # YOUR CODE HERE
attack1_password = ""

print("=== Attack 1: Bypass login ===")
login_vulnerable(db, attack1_username, attack1_password)


In [ ]:
# ========================================
# TODO: Attack 2 — Steal credit card numbers
# ========================================
# Goal: Extract all credit card numbers from the database.

attack2_username = ""  # YOUR CODE HERE
attack2_password = ""

print("=== Attack 2: Steal credit cards ===")
login_vulnerable(db, attack2_username, attack2_password)


In [ ]:
# ========================================
# TODO: Attack 3 — Drop the users table
# ========================================
# Goal: Destroy the entire users table.
#
# WARNING: This will actually delete the table from our in-memory database!
#
# SQLite's execute() doesn't allow multiple statements by default.
# We will use executescript() to simulate a more permissive backend.

attack3_input = ""  # YOUR CODE HERE

print("=== Attack 3: Drop the table ===")
query = f"SELECT * FROM users WHERE username = '{attack3_input}'"
print(f"Generated query:\n  {query}\n")
try:
    db.executescript(query)
    print("Query executed.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Check if the table still exists
try:
    db.execute("SELECT count(*) FROM users")
    print("Table 'users' still exists.")
except:
    print("Table 'users' has been DESTROYED!")

# Recreate for the rest of the exercises
db.close()
db = setup_database()
print("\n(Database recreated for remaining exercises)")


#### Task 1b: Fix the Vulnerability

Rewrite the login function using parameterized queries.

With parameterized queries, you pass the user input as separate parameters to the database engine, not as part of the SQL string. The database treats them as data, never as code, so injection is impossible.

In SQLite (Python), parameterized queries use `?` placeholders.

In [ ]:
# ========================================
# TODO: Implement a SAFE login function
# ========================================

def login_safe(conn, username, password):
    # YOUR CODE HERE
    raise NotImplementedError("Replace this with your implementation")


# Test: normal login should still work
success, rows = login_safe(db, "alice", "password123")
print(f"Normal login: {'Success' if success else 'Failed'}\n")

# Test: injection attack should now FAIL
print("=== Injection attempt ===")
success, rows = login_safe(db, "' OR 1=1 --", "anything")
print(f"Result: {'Success (still vulnerable!)' if success else 'Failed (Good, attack blocked!)'}")


#### Task 1c

The `login_vulnerable` function's database connection has full privileges: it can SELECT, INSERT,
UPDATE, DELETE, and DROP any table. Even if you can't change the code, you could limit the damage by
restricting the database account's permissions.

**Question:** If you could only change the database permissions, what specific
SQL privileges would you grant to the backend's database account, and why?

**Your answer:**



---

### Exercise 2: Password Storage & Cracking

When users create accounts, the backend must store something that lets it verify
their password later. The question is, what to store. As discussed, there's a
progression from terrible (plaintext) to good (bcrypt with salt):

1. Plaintext: store the password directly. If the database leaks, every password is exposed instantly.
2. Unsalted hash (MD5): store `MD5(password)`. Better, but vulnerable to rainbow tables (precomputed hash -> password lookup tables).
3. Salted hash (SHA-256): store `SHA256(salt + password)` and the salt. Each user gets a random salt, so rainbow tables don't work. But SHA-256 is fast, so brute force is still feasible.
4. bcrypt salted hash: each hash takes ~100ms instead of ~1μs. This makes brute force impractical.

In this exercise, we will simulate a database breach and try to crack passwords stored each way. We will observe why the storage method matters.


In [ ]:
# ========================================
# Exercise 2 Setup
# ========================================
# DO NOT MODIFY

import hashlib
import time
import bcrypt as bcrypt_lib
import os

# A small wordlist of common passwords (a real attacker would use millions)
COMMON_PASSWORDS = [
    "123456", "password", "password123", "12345678", "qwerty", "abc123",
    "monkey", "dragon", "letmein", "trustno1", "baseball", "iloveyou",
    "master", "sunshine", "ashley", "michael", "shadow", "123123",
    "654321", "superman", "qazwsx", "welcome", "welcome1", "football",
    "jesus", "ninja", "mustang", "access", "flower", "whatever",
    "charlie", "donald", "batman", "starwars", "hello", "freedom",
    "thunder", "matrix", "hunter2", "ilovecats", "dragon99", "P@ssw0rd",
    "qwerty2024", "securePass!1", "letmein2024", "admin123", "root",
    "toor", "pass", "guest", "test", "love", "god", "secret",
]

# The "secret" password we'll try to crack (same for all methods, to make comparison fair)
SECRET_PASSWORD = "dragon99"

# Method 1: Plaintext storage
stored_plaintext = SECRET_PASSWORD

# Method 2: Unsalted MD5
stored_md5 = hashlib.md5(SECRET_PASSWORD.encode()).hexdigest()

# Method 3: Salted SHA-256
salt = os.urandom(16)  # 16 random bytes
stored_salt = salt
stored_salted_sha256 = hashlib.sha256(salt + SECRET_PASSWORD.encode()).hexdigest()

# Method 4: bcrypt (with built-in salt, cost factor 12)
stored_bcrypt = bcrypt_lib.hashpw(SECRET_PASSWORD.encode(), bcrypt_lib.gensalt(rounds=12))

print("Simulated 'leaked database' created.")
print(f"\nWhat an attacker sees after the breach:")
print(f"  Plaintext entry:    {stored_plaintext}")
print(f"  MD5 hash:           {stored_md5}")
print(f"  SHA-256 + salt:     {stored_salted_sha256}")
print(f"    (salt):           {stored_salt.hex()}")
print(f"  bcrypt hash:        {stored_bcrypt.decode()}")
print(f"\nThe attacker's wordlist has {len(COMMON_PASSWORDS)} common passwords.")


In [ ]:
# ========================================
# TODO: Task 2a
# ========================================

def crack_plaintext(stored_value, wordlist):
    """
    'Crack' a plaintext-stored password.

    Check if the stored value is in the wordlist.
    Return the password if found, None otherwise.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


def crack_md5(stored_hash, wordlist):
    """
    Crack an unsalted MD5 hash.

    For each word in the wordlist comparevhashlib.md5(word.encode()).hexdigest()
    to stored_hash. Return the word if found, None otherwise.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# --- Run the attacks ---
print("=== Cracking plaintext ===")
start = time.time()
result = crack_plaintext(stored_plaintext, COMMON_PASSWORDS)
elapsed = time.time() - start
print(f"  Found: {result} in {elapsed*1000:.2f} ms\n")

print("=== Cracking MD5 (no salt) ===")
start = time.time()
result = crack_md5(stored_md5, COMMON_PASSWORDS)
elapsed = time.time() - start
print(f"  Found: {result} in {elapsed*1000:.2f} ms")


In [ ]:
# ========================================
# TODO: Task 2b
# ========================================

def crack_salted_sha256(stored_hash, salt, wordlist):
    """
    Crack a salted SHA-256 hash.

    Similar to crack_md5. Use hashlib.sha256() instead, and prepend the salt.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


def crack_bcrypt(stored_hash, wordlist):
    """
    Crack a bcrypt hash.

    Similar to crack_md5. Use bcrypt_lib.checkpw() instead. Salt is integrated.

    NOTE: bcrypt is intentionally slow. This will take longer!
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# --- Run the attacks ---
print("=== Cracking salted SHA-256 ===")
start = time.time()
result = crack_salted_sha256(stored_salted_sha256, stored_salt, COMMON_PASSWORDS)
elapsed = time.time() - start
print(f"  Found: {result} in {elapsed*1000:.2f} ms\n")

print("=== Cracking bcrypt (this will be slow — watch the time!) ===")
start = time.time()
result = crack_bcrypt(stored_bcrypt, COMMON_PASSWORDS)
elapsed = time.time() - start
print(f"  Found: {result} in {elapsed*1000:.2f} ms")
print(f"  That's {elapsed:.1f} seconds!")


#### Task 2c

**Your answer:** Fill in the table and answer the questions below.

| Storage Method | Time to Crack | Why? |
|---|---|---|
| Plaintext | ??? ms | |
| MD5 (no salt) | ??? ms | |
| Salted SHA-256 | ??? ms | |
| bcrypt | ??? ms | |

1. Why is bcrypt so much slower than SHA-256?

2. Our wordlist only has ~50 passwords. A real attacker might use a list of 10 billion passwords. Roughly how long would it take to brute-force bcrypt with 10 billion guesses?



---

### Exercise 3: De-anonymization & k-Anonymity

When organizations release datasets for research or public use, they typically
remove obvious identifiers like names and SSNs. But this is often not enough. Quasi-identifiers, fields like zip code, date of birth, and gender, don't identify someone individually, but in combination they can uniquely pinpoint a person. The Netflix Prize de-anonymization worked similarly: even though Netflix removed usernames, researchers cross-referenced movie ratings with public IMDB profiles to identify users.

In this exercise, you'll reproduce a simplified version of these attacks, then apply k-anonymity to defend against them, and discover its limitations.


In [ ]:
# ========================================
# Exercise 3 Setup
# ========================================
# DO NOT MODIFY

import pandas as pd
import numpy as np

np.random.seed(4121)  # For reproducibility

# --- Generate overlapping populations ---
n_total = 200
n_overlap = 80  # People who appear in BOTH datasets

# Quasi-identifier pools
zip_codes = ["10025", "10027", "10030", "10032", "10033", "10040", "10031", "10029"]
genders = ["M", "F"]
diagnoses = ["Flu", "Diabetes", "Heart Disease", "Cancer", "Anxiety", "Migraine", "Asthma", "HIV"]
parties = ["Democrat", "Republican", "Independent", "Green"]

# Generate birth dates (ages 20-80)
def random_birthdate():
    year = np.random.randint(1945, 2005)
    month = np.random.randint(1, 13)
    day = np.random.randint(1, 29)
    return f"{year}-{month:02d}-{day:02d}"

# Create the "overlap" population — these people appear in both datasets
overlap_records = []
for i in range(n_overlap):
    overlap_records.append({
        "full_name": f"Person_{i:03d}",
        "zip_code": np.random.choice(zip_codes),
        "birth_date": random_birthdate(),
        "gender": np.random.choice(genders),
        "diagnosis": np.random.choice(diagnoses),
        "party": np.random.choice(parties),
    })

# Anonyized medical dataset released
# Contains the overlap population + extra patients
medical_records = []
for rec in overlap_records:
    medical_records.append({
        "patient_id": np.random.randint(100000, 999999),  # Random fake ID (replacing real name)
        "zip_code": rec["zip_code"],
        "birth_date": rec["birth_date"],
        "gender": rec["gender"],
        "diagnosis": rec["diagnosis"],
    })
# Add non-overlapping patients
for i in range(n_total - n_overlap):
    medical_records.append({
        "patient_id": np.random.randint(100000, 999999),
        "zip_code": np.random.choice(zip_codes),
        "birth_date": random_birthdate(),
        "gender": np.random.choice(genders),
        "diagnosis": np.random.choice(diagnoses),
    })
np.random.shuffle(medical_records)
df_medical = pd.DataFrame(medical_records)

# Publicly available voter registration dataset
# Contains the overlap population + extra voters
voter_records = []
for rec in overlap_records:
    voter_records.append({
        "full_name": rec["full_name"],
        "zip_code": rec["zip_code"],
        "birth_date": rec["birth_date"],
        "gender": rec["gender"],
        "party": rec["party"],
    })
# Add non-overlapping voters
for i in range(150 - n_overlap):
    voter_records.append({
        "full_name": f"Voter_{i:03d}",
        "zip_code": np.random.choice(zip_codes),
        "birth_date": random_birthdate(),
        "gender": np.random.choice(genders),
        "party": np.random.choice(parties),
    })
np.random.shuffle(voter_records)
df_voter = pd.DataFrame(voter_records)

print("Two datasets created:\n")
print(f"Medical dataset (anonymized): {len(df_medical)} rows")
print(f"  Columns: {list(df_medical.columns)}")
print(df_medical.head())
print(f"\nVoter registration (public): {len(df_voter)} rows")
print(f"  Columns: {list(df_voter.columns)}")
print(df_voter.head())
print(f"\nNote: {n_overlap} individuals appear in BOTH datasets (but you don't know which ones yet).")


In [ ]:
# ========================================
# TODO: Task 3a — Re-identification attack
# ========================================
# Goal: Link real names from the voter dataset to diagnoses in the medical dataset.
#
# The quasi-identifiers that appear in BOTH datasets are: zip_code, birth_date, gender.
# By joining (merging) on these columns, you can match anonymous patient records
# to named voter records — revealing who has which diagnosis.
#
# Count how many individuals you successfully re-identified (rows in the merged result).
# Display the first 10 re-identified records, showing name + diagnosis.
#
# Hint:
#   Use: pd.merge(df_medical, df_voter, on=[...list of shared columns...])

# YOUR CODE HERE
raise NotImplementedError()


#### Task 3b: Apply k-Anonymity

The attack worked because quasi-identifier combinations were too specific — many were unique
to a single person. k-Anonymity says that every combination of quasi-identifiers must appear
in at least *k* rows. If k=5, then any individual is hidden among at least 4 other people with
the same quasi-identifier values.

To achieve this, we generalize the data, i.e., reduce its precision:
- Zip codes: truncate to first 3 digits
- Birth dates: convert to 10-year age ranges (e.g., "1985-03-14" -> "40-49")

Write a function that applies these generalizations, then re-run the attack and see how the re-identification rate changes.


In [ ]:
# ========================================
# TODO: Task 3b — Generalize for k-anonymity
# ========================================

def generalize(df):
    """
    Apply k-anonymity generalizations to a DataFrame.

    Create a COPY of the input (don't modify the original!).
    Return the generalized DataFrame.
    """
    df_copy = df.copy()
    # YOUR CODE HERE
    raise NotImplementedError()


# Generalize the medical dataset
df_medical_gen = generalize(df_medical)
print("Generalized medical dataset (first 10 rows):")
print(df_medical_gen.head(10).to_string(index=False))

# Generalize the voter dataset (attacker must generalize their data the same way)
df_voter_gen = generalize(df_voter)

# Re-run the attack on generalized data. Copy from 3a

# Check: how many quasi-identifier groups have only 1 medical record? (those are still uniquely identifiable)
group_sizes = df_medical_gen.groupby(["zip_code", "birth_date", "gender"]).size()
unique_groups = (group_sizes == 1).sum()
print(f"\nQuasi-identifier groups with only 1 record: {unique_groups} out of {len(group_sizes)}")
print(f"Groups with 5+ records (k≥5): {(group_sizes >= 5).sum()} out of {len(group_sizes)}")


#### Task 3c: The Homogeneity Attack — k-Anonymity Isn't Enough

Even with k-anonymity, there's a subtle problem. If all k records in a group share the
same sensitive value (e.g., all have the same diagnosis), then the attacker doesn't need
to identify the specific individual — they already know the diagnosis.

This is called the homogeneity attack, and it's the motivation for **l-diversity**
(each group must have at least *l* distinct sensitive values).

Find groups in the generalized medical dataset where all records have the same diagnosis.


In [ ]:
# ========================================
# TODO: Task 3c — Find the homogeneity attack
# ========================================
# Goal: Find groups where k-anonymity holds (multiple records per group)
# but ALL records in the group have the SAME diagnosis.
#
# Steps:
#   1. Group df_medical_gen by the quasi-identifiers
#   2. For each group, get group size, number of unique diagnoses, and the diagnosis values
#   3. Filter to groups where size > 1 AND number of unique diagnoses is 1

# YOUR CODE HERE
raise NotImplementedError()


#### Task 3d: Short Answer — Connecting to the Netflix Case

**Question:** Netflix believed that removing usernames and slightly perturbing ratings was sufficient
to anonymize their dataset. Based on what you just did in this exercise, why wasn't it?
What is the general principle at work?

**Your answer:**



---
## Section 2: Systems for ML

---

### Exercise 4: GPU Memory Budget Calculator

GPU memory is the most common constraint in ML. Before you start training or
serving a model, you need to know whether it fits. GPU memory (HBM) is limited and expensive: 80 GB on an H100, 192 GB on a B200. Every byte matters. During training, you need to hold:

1. Model weights — the parameters themselves
2. Gradients — same size as weights (one gradient per parameter)
3. Optimizer states — Adam stores 2 extra values per parameter (momentum + variance), in FP32
4. Activations — intermediate values saved during forward pass for use in backward pass

During inference, you need:
1. All model weights
2. KV cache — stored key/value tensors from previous tokens (grows with sequence length)

In this exercise, you'll build functions to calculate each component and answer "does it fit?" questions for models like Llama 3 8B and 70B.


In [ ]:
# ========================================
# Exercise 4 Setup — model configs and GPU specs
# ========================================
# DO NOT MODIFY

# Model configurations
MODELS = {
    "Llama-3-8B": {
        "params": 8e9,        # 8 billion parameters
        "hidden_dim": 4096,
        "num_layers": 32,
        "num_heads": 32,
        "head_dim": 128,      # hidden_dim / num_heads
    },
    "Llama-3-70B": {
        "params": 70e9,       # 70 billion parameters
        "hidden_dim": 8192,
        "num_layers": 80,
        "num_heads": 64,
        "head_dim": 128,
    },
}

# Precision: bytes per parameter
PRECISIONS = {
    "FP32":  4,
    "FP16":  2,
    "BF16":  2,
    "INT8":  1,
    "INT4":  0.5,
}

# GPU specs (memory in GB)
GPUS = {
    "T4":   {"memory_gb": 16,  "cost_per_hr": 0.50},
    "A10G": {"memory_gb": 24,  "cost_per_hr": 1.50},
    "A100": {"memory_gb": 80,  "cost_per_hr": 15.00},
    "H100": {"memory_gb": 80,  "cost_per_hr": 30.00},
    "B200": {"memory_gb": 192, "cost_per_hr": 50.00},
}

print("Model configs and GPU specs loaded.")
print(f"\nModels: {list(MODELS.keys())}")
print(f"Precisions: {list(PRECISIONS.keys())}")
print(f"GPUs: {list(GPUS.keys())}")


In [ ]:
# ========================================
# TODO: Task 4a — Weight memory
# ========================================

def calc_weight_memory_gb(num_params, bytes_per_param):
    """
    Calculate memory needed to store model weights.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# Fill in the table: weight memory for each model × precision combination
print("Weight Memory (GB):")
print(f"{'Model':<15} {'FP32':>8} {'FP16':>8} {'BF16':>8} {'INT8':>8} {'INT4':>8}")
print("-" * 63)
for model_name, config in MODELS.items():
    row = f"{model_name:<15}"
    for prec_name, bpp in PRECISIONS.items():
        mem = calc_weight_memory_gb(config["params"], bpp)
        row += f" {mem:>7.1f}"
    print(row)


In [ ]:
# ========================================
# TODO: Task 4b — Training memory
# ========================================

def calc_training_memory_gb(num_params, precision="BF16"):
    """
    Calculate TOTAL GPU memory needed for training with Adam optimizer.

    Hints:
    Everything listed here must be in memory simultaneously:
      1. Weights: num_params × bytes_per_param
      2. Gradients: same size as weights (one gradient per parameter, same precision)
      3. Optimizer states (Adam): 2 copies per parameter, ALWAYS in FP32 (4 bytes each)
         - First moment (momentum): num_params × 4 bytes
         - Second moment (variance): num_params × 4 bytes

    IMPORTANT: Even when training in FP16/BF16, Adam's optimizer states are kept in FP32.
    This is why the "4x rule" is approximate — it's exact for FP32, but for FP16/BF16 the
    ratio is actually higher than 4x because optimizer states don't shrink.
    """

    # YOUR CODE HERE
    raise NotImplementedError()


# Compare training memory at different precisions
print("Training Memory (GB) — weights + gradients + optimizer states:")
print(f"{'Model':<15} {'FP32':>10} {'FP16':>10} {'BF16':>10}")
print("-" * 47)
for model_name, config in MODELS.items():
    row = f"{model_name:<15}"
    for prec in ["FP32", "FP16", "BF16"]:
        mem = calc_training_memory_gb(config["params"], prec)
        row += f" {mem:>9.1f}"
    print(row)

In [ ]:
# ========================================
# TODO: Task 4c — Activation memory
# ========================================

def calc_activation_memory_gb(batch_size, seq_len, hidden_dim, num_layers, precision="BF16"):
    """
    Estimate activation memory during training.

    Hints:
    During the forward pass, intermediate values (activations) are saved at each
    layer so they can be used during the backward pass to compute gradients.

    activation_memory ≈ batch_size × seq_len × hidden_dim × num_layers × bytes_per_element

    This counts roughly one activation tensor per layer. Real models save more
    (attention scores, FFN intermediates, etc.), so actual memory is 2-4x higher,
    but this gives the right order of magnitude.

    """
    # YOUR CODE HERE
    raise NotImplementedError()


# Find the maximum batch size that fits on an 80GB A100 for Llama 8B in BF16
config = MODELS["Llama-3-8B"]
base_training_mem = calc_training_memory_gb(config["params"], "BF16")
gpu_memory = GPUS["A100"]["memory_gb"]

print(f"Llama-3-8B training memory (no activations): {base_training_mem:.1f} GB")
print(f"A100 GPU memory: {gpu_memory} GB")
print(f"Memory available for activations: {gpu_memory - base_training_mem:.1f} GB\n")

print(f"{'Batch Size':>10} {'Activation Mem (GB)':>20} {'Total (GB)':>12} {'Fits on A100?':>14}")
print("-" * 60)

max_batch = 0
for bs in [1, 2, 4, 8, 16, 32, 64]:
    act_mem = calc_activation_memory_gb(bs, 2048, config["hidden_dim"], config["num_layers"], "BF16")
    total = base_training_mem + act_mem
    fits = "Yes" if total <= gpu_memory else "No"
    if total <= gpu_memory:
        max_batch = bs
    print(f"{bs:>10} {act_mem:>20.1f} {total:>12.1f} {fits:>14}")

print(f"\nMaximum batch size on A100: {max_batch}")


In [ ]:
# ========================================
# TODO: Task 4d — KV cache for inference
# ========================================

def calc_kv_cache_memory_gb(seq_len, hidden_dim, num_layers, precision="FP16"):
    """
    Calculate KV cache memory for a SINGLE user request during inference.

    During text generation, the model caches the Key and Value tensors from
    all previous tokens at every layer, so it doesn't have to recompute them.

    Hint:
        kv_cache = 2 × seq_len × hidden_dim × num_layers × bytes_per_element
        (The "2" is for Key + Value — each is the same size)
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# Calculate KV cache for Llama 70B
config_70b = MODELS["Llama-3-70B"]
kv_per_request = calc_kv_cache_memory_gb(
    seq_len=4096,
    hidden_dim=config_70b["hidden_dim"],
    num_layers=config_70b["num_layers"],
    precision="FP16"
)
print(f"KV cache per request (Llama 70B, seq_len=4096, FP16): {kv_per_request:.2f} GB")

# How many simultaneous users on an H100?
model_weight_mem = calc_weight_memory_gb(config_70b["params"], PRECISIONS["INT8"])
available = GPUS["H100"]["memory_gb"] - model_weight_mem
max_users = int(available / kv_per_request)

print(f"\nLlama 70B weight memory (INT8): {model_weight_mem:.1f} GB")
print(f"H100 memory: {GPUS['H100']['memory_gb']} GB")
print(f"Memory available for KV caches: {available:.1f} GB")
print(f"\n→ Maximum simultaneous users: {max_users}")


#### Task 4e: Scenario — Self-Host vs. API?

Your startup wants to deploy Llama 70B for a customer support chatbot. You expect 5,000 requests/day, with an average of 500 input tokens + 200 output tokens per request.

Option A: Self-host on H100**
- H100 costs $30/hr
- Assume INT8 quantization, throughput of ~100 tokens/sec (batched)
- You need the GPU running 24/7 (even when idle)

Option B: Use an API (e.g., Anthropic, OpenAI)
- Input tokens: \$10 per million tokens
- Output tokens: \$30 per million tokens

**Question:** Calculate the monthly cost for each option. Which would you choose and why? Why does quantization matter?

**Your answer:**

---

### Exercise 5: Profiling a Training Loop

Exercise 4 tells us what *should* fit. But real-world performance depends on many factors the math doesn't capture: data loading speed, GPU kernel efficiency, memory fragmentation, and more.

In this exercise, you will train a real model on a real GPU and observe how batch size, mixed precision, and data loading settings affect throughput and memory usage. You'll connect the empirical results back to the theory.

**This exercise requires a GPU.** In Colab, go to Runtime > Change runtime type > T4 GPU.

In [ ]:
# ========================================
# Exercise 5 Setup — model and data
# ========================================
# DO NOT MODIFY

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import time

# Check GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    device = torch.device("cpu")
    print("No GPU available. Running on CPU (results will be much slower).")
    print("Go to Runtime → Change runtime type → T4 GPU")

# Download CIFAR-10 (small dataset, fast to download)
transform = transforms.Compose([
    transforms.Resize(224),  # ResNet expects 224x224
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
print(f"\nCIFAR-10 loaded: {len(dataset)} training images")


def train_one_epoch(model, dataloader, optimizer, device, use_amp=False):
    """
    Train for one epoch and return throughput + peak memory.
    This function is provided — you'll call it with different settings.
    """
    model.train()
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    total_images = 0
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    for batch_idx, (images, labels) in enumerate(dataloader):
        if batch_idx >= 20:  # Only run 20 batches (enough to measure, keeps it fast)
            break

        images, labels = images.to(device), labels.to(device)

        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_images += images.size(0)

    elapsed = time.time() - start_time
    throughput = total_images / elapsed
    peak_memory_gb = torch.cuda.max_memory_allocated() / (1024**3)

    return throughput, peak_memory_gb


print("\nTraining function ready.")


In [ ]:
# ========================================
# TODO: Task 5a — Batch size sweep
# ========================================
# Goal: Measure throughput (images/sec) and peak GPU memory at different batch sizes.
# If a batch size causes an OOM error, catch it and record "OOM".

batch_sizes = [16, 32, 64, 128, 256, 512, 1024, 2048]
results_5a = []

for bs in batch_sizes:
    print(f"\nBatch size = {bs}...", end=" ")
    try:
        # YOUR CODE HERE
        # 1. Create dataloader: DataLoader(dataset, batch_size=bs, shuffle=True, num_workers=2)
        # 2. Create model: torchvision.models.resnet18(num_classes=10).to(device)
        # 3. Create optimizer: torch.optim.SGD(model.parameters(), lr=0.01)
        # 4. Call: throughput, peak_mem = train_one_epoch(model, dataloader, optimizer, device, use_amp=False)
        # 5. Append results: results_5a.append({"batch_size": bs, "throughput": throughput, "peak_memory_gb": peak_mem})
        # 6. Clean up: del model, optimizer; torch.cuda.empty_cache()

        raise NotImplementedError()

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("OOM!")
            results_5a.append({"batch_size": bs, "throughput": 0, "peak_memory_gb": float('inf')})
            torch.cuda.empty_cache()
        else:
            raise

# Display results
import pandas as pd
df_results = pd.DataFrame(results_5a)
print("\n\n=== Batch Size Sweep Results ===")
print(df_results.to_string(index=False))


In [ ]:
# ========================================
# TODO: Task 5b — Mixed precision (AMP)
# ========================================
# Goal: Re-run the batch size sweep with Automatic Mixed Precision (AMP) enabled.
#
# AMP uses FP16 for most operations and FP32 only where needed.
# This should:
#   - Reduce peak memory (FP16 activations are half the size)
#   - Increase throughput (tensor cores are faster with FP16)
#   - Allow larger batch sizes before OOM
#
# The ONLY change from Task 5a: pass use_amp=True to train_one_epoch().
# Try the same batch sizes PLUS 512 (which probably OOM'd before).

batch_sizes_amp = [16, 32, 64, 128, 256, 512, 1024, 2048]
results_5b = []

for bs in batch_sizes_amp:
    print(f"\nBatch size = {bs} (AMP)...", end=" ")
    try:
        # YOUR CODE HERE
        # Same as 5a, but pass use_amp=True to train_one_epoch
        raise NotImplementedError()

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("OOM!")
            results_5b.append({"batch_size": bs, "throughput": 0, "peak_memory_gb": float('inf')})
            torch.cuda.empty_cache()
        else:
            raise

df_results_amp = pd.DataFrame(results_5b)
print("\n\n=== Batch Size Sweep with AMP ===")
print(df_results_amp.to_string(index=False))

# Compare: what's the largest batch size that fits with vs. without AMP?


In [ ]:
# ========================================
# TODO: Task 5c — Data loading bottleneck
# ========================================
# Goal: Measure how num_workers affects throughput at a fixed batch size.
#
# num_workers controls how many CPU processes load and preprocess data in parallel.
#   - num_workers=0: data loading happens in the main process (slowest)
#   - num_workers=2, 4, 8: parallel data loading (faster, up to a point)
#
# Use batch_size=128 and use_amp=True for all runs.

worker_counts = [0, 2, 4, 8]
results_5c = []

for nw in worker_counts:
    print(f"\nnum_workers = {nw}...", end=" ")

    # YOUR CODE HERE
    # Same as before, but vary num_workers in the DataLoader and keep batch_size=128, use_amp=True
    raise NotImplementedError()

import pandas as pd
df_workers = pd.DataFrame(results_5c)
print("\n\n=== Data Loading: num_workers Sweep ===")
print(df_workers.to_string(index=False))


#### Task 5d: Analysis

Answer the following questions based on your experimental results.

**Q1:** At what batch size did you hit OOM without AMP? With AMP? How does this relate
to the memory formulas from Exercise 4? (Remember: AMP uses FP16 for activations, which
are the component that scales with batch size.)

**Your answer:**



**Q2:** At what `num_workers` value did you see diminishing returns in throughput?
Why does increasing workers beyond that point not help? (Hint: think about whether
the bottleneck is on the CPU or GPU side.)

**Your answer:**



**Q3:** Using your best throughput number (largest batch size + AMP + optimal workers),
estimate how long it would take to train ResNet-18 on the full CIFAR-10 training set
(50,000 images) for 10 epochs. What would this cost on a T4 at $0.50/hr?

**Your answer:**

